# Is the streamer using an aimbot?
MA206 Lesson 21, Problem 1 companion. Run each cell in order.

**Honest players at his rank:** average reaction time 200 ms, SD 50 ms. **The streamer:** 400 fights, average 197.5 ms.

Aimbot example: https://youtu.be/vbMWtmzpCog?si=Q-68yPff4vI1fwKE

In [ ]:
library(tidyverse)
set.seed(206)
options(repr.plot.width = 10, repr.plot.height = 4.5)

## 1. One honest player, one fight
Nobody reacts faster than about 150 ms, but a distracted moment can take 400. Single reaction times are **strongly right-skewed** — nothing like a bell.

In [ ]:
pop <- tibble(x = seq(150, 500, by = 1)) |>
  mutate(density = dexp(x - 150, rate = 1/50))

ggplot(pop, aes(x, density)) +
  geom_area(fill = "grey80") +
  geom_line(linewidth = 1) +
  geom_vline(xintercept = 200, linetype = "dashed") +
  annotate("text", x = 207, y = 0.018, label = "mean = 200 ms", hjust = 0, size = 5) +
  labs(title = "One honest player, one fight",
       subtitle = "Single reaction times are strongly right-skewed",
       x = "reaction time (ms)", y = "density") +
  theme_minimal(base_size = 15) +
  theme(plot.background = element_rect(fill = "white", color = NA))

## 2. Simulate 10,000 honest players, 400 fights each
Each column of the matrix is one player. `colMeans` gives each player's average.

In [ ]:
n    <- 400
reps <- 10000
reactions <- matrix(150 + rexp(n * reps, rate = 1/50), nrow = n)
xbar <- colMeans(reactions)
sims <- tibble(xbar = xbar)

mean(xbar)   # about 200
sd(xbar)     # about 2.5 = 50 / sqrt(400)

## 3. The averages: the skew is gone
The curve is the CLT's prediction, N(200, 2.5²), drawn before looking at the simulation.

In [ ]:
bw   <- 0.5
lims <- c(190, 210)

ggplot(sims, aes(xbar)) +
  geom_histogram(aes(y = after_stat(count) / (reps * bw)),
                 binwidth = bw, boundary = 197.5,
                 fill = "grey70", color = "white") +
  geom_function(fun = dnorm, args = list(mean = 200, sd = 2.5), linewidth = 1) +
  coord_cartesian(xlim = lims) +
  labs(title = "Averages of 10,000 honest players (400 fights each)",
       subtitle = "Curve: the CLT's prediction, N(200, 2.5^2)",
       x = "average reaction time (ms)", y = "density") +
  theme_minimal(base_size = 15) +
  theme(plot.background = element_rect(fill = "white", color = NA))

## 4. The P-value: how often does an honest player average 197.5 ms or faster?
Same plot, with the honest players who did at least as well as the streamer shaded red.

In [ ]:
observed <- 197.5
p_sim  <- mean(xbar <= observed)      # simulation
p_norm <- pnorm(observed, 200, 2.5)   # normal table
c(simulation = p_sim, normal = p_norm)

ggplot(sims, aes(xbar)) +
  geom_histogram(aes(y = after_stat(count) / (reps * bw),
                     fill = xbar <= observed),
                 binwidth = bw, boundary = observed, color = "white") +
  scale_fill_manual(values = c(`TRUE` = "firebrick", `FALSE` = "grey70"),
                    guide = "none") +
  geom_function(fun = dnorm, args = list(mean = 200, sd = 2.5), linewidth = 1) +
  geom_vline(xintercept = observed, linewidth = 1) +
  annotate("text", x = observed - 0.3, y = 0.15, hjust = 1, size = 5,
           label = paste0("streamer's average: ", observed, " ms\n",
                          "red area = P-value = ", round(p_sim, 3))) +
  coord_cartesian(xlim = lims) +
  labs(title = "How often does an HONEST player average this fast?",
       subtitle = "Red = honest players who averaged 197.5 ms or faster",
       x = "average reaction time (ms)", y = "density") +
  theme_minimal(base_size = 15) +
  theme(plot.background = element_rect(fill = "white", color = NA))

## Try it
Change `observed` to 195 or 192.5 and rerun the last cell. Watch the red area shrink — that is the evidence getting stronger.